# A Brief Exploration of LiDAR Processing in Python

In [ ]:
import sys
!{sys.executable} -m pip install whitebox-workflows
import whitebox 
import whitebox_workflows as wbw
import geemap 
import rioxarray as rxr 
import rasterio 

# Ensure pkg_resources is available (comes from setuptools)
!{sys.executable} -m pip install -q setuptools
import matplotlib.pyplot as plt 
import numpy as np 
import geopandas as gpd 
from rasterio.plot import plotting_extent

In [ ]:
#Instantiate whitebox tools 
wbt = whitebox.WhiteboxTools()

In [ ]:
wbt.lidar_info(
    i = "/Users/tomasmejia/coding/Lidar/data/2013_BLDR_flood_2013100814_471000_4428000.laz",
    output = "/Users/tomasmejia/coding/Lidar/outputs/colorad_laz_info.html"
)


## Plotting Values into graphs

In [ ]:
#plotting the Elevation values of the Colorado .laz file 
wbt.lidar_histogram(
    i = "/Users/tomasmejia/coding/Lidar/data/2013_BLDR_flood_2013100814_471000_4428000.laz",
    output = "/Users/tomasmejia/coding/Lidar/outputs/colorad_elev_hist.html", 
    parameter="elevation", #controls what attribute of the LiDAR point cloud gets plotted on the histogram's x-axis.
    clip=0 #This is a percentage value that clips the upper and lower tails of the frequency distribution.
)  

In [ ]:
#Plot a histogram of the classes within our Colorado .laz fiile
wbt.lidar_histogram(
    i = "/Users/tomasmejia/coding/Lidar/data/2013_BLDR_flood_2013100814_471000_4428000.laz",
    output = "/Users/tomasmejia/coding/Lidar/outputs/colorado_class_hist.html",
    parameter = "class",
    clip = 0    
)

## Creating DEM models using IDW

In [ ]:
#Creating a Digital Elevation Model (DEM) using Inverse Distance Weighting (IDW) interpolation method. 
wbt.lidar_idw_interpolation(
    i = "/Users/tomasmejia/coding/Lidar/data/2013_BLDR_flood_2013100814_471000_4428000.laz",
    output = "/Users/tomasmejia/coding/Lidar/outputs/idw_dem.tif",
    parameter = "elevation",
    returns = "last",
    exclude_cls="1, 5, 6" #we want to exclude the classes of Unclassified, High Vegetation, and Building respectively
)

In [ ]:
#Assigning Coordinate Reference System (CRS) to the DEM
#we will be utilizing geemap package for this 
geemap.add_crs("/Users/tomasmejia/coding/Lidar/outputs/idw_dem.tif",
                epsg=32613) # This CRS is based on one of the shapefiles withing the downloaded data

In [ ]:
#Read the raster using rioxarray 
idw_dem = rxr.open_rasterio("/Users/tomasmejia/coding/Lidar/outputs/idw_dem.tif", masked=True)

In [ ]:
#Displaying the raster
fig, ax = plt.subplots(figsize=(8,8))
im = idw_dem.plot(ax=ax, cmap = "RdYlGn", add_colorbar = True)
ax.set_title("Lidar DEM using IDW")
plt.colorbar(im, ax=ax)

plt.show()

## Creating DEM using TIN 